In [1]:
# installing dependencies
%pip install -r requirements.txt -q


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# importing the required dependencies
import os
import json
from dotenv import load_dotenv

from llama_index.llms.groq import Groq
from llama_index.llms.openai import OpenAI

In [4]:
# load env variables
load_dotenv()

LLM_A = os.getenv("LLM_A")
LLM_B = os.getenv("LLM_B")
JUDGE_LLM = os.getenv("JUDGE_LLM")

In [8]:
# initialize llms
llm_a = Groq(model=LLM_A, temperature=0.0)
llm_b = Groq(model=LLM_B, temperature=0.0)

judge_llm = OpenAI(model=JUDGE_LLM, temperature=0.0)

In [ ]:
# function: get llm response
def get_llm_response(llm, user_prompt):
    response = llm.complete(user_prompt)
    return response.text.strip()

In [15]:
# function: judge two responses
def judge_responses(judge_llm, user_prompt, answer_a, answer_b):
    judge_prompt = f"""
You are an impartial and strict evaluator.

Your job is to compare two answers to the SAME user question
and decide which one is better based on the criteria below.

USER QUESTION:
{user_prompt}

ANSWER A:
{answer_a}

ANSWER B:
{answer_b}

Evaluate using ONLY these criteria (in order of priority):

1. Correctness
   - Are the facts accurate?
   - Are there any wrong or misleading statements?

2. Completeness
   - Does the answer fully address the question?
   - Does it miss important points?

3. Clarity
   - Is the explanation easy to understand for a beginner?
   - Is it well structured and readable?

4. Safety and Best Practices
   - Does it avoid harmful, insecure, or unsafe guidance?
   - For cloud/dev topics, does it follow best practices?

Decision Rules:
- If one answer is clearly better overall, choose that answer.
- If both answers have issues, choose the LESS WRONG one.
- If both answers are equally good or equally bad, choose "tie".
- Do NOT judge based on writing style alone.

Return ONLY valid JSON in this exact format:
{{
  "winner": "A" or "B" or "tie",
  "reason": "short explanation mentioning the key deciding factors"
}}
"""
    judge_response = judge_llm.complete(judge_prompt)
    return json.loads(judge_response.text.strip())


In [16]:
# smaple queries
test_prompts = [
    "Explain AWS IAM in simple terms",
    "Difference between Security Group and NACL",
    "What is Docker and why is it used",
    "Explain CI/CD for a beginner",
    "What is Kubernetes in simple words"
]

In [17]:
score_a = 0
score_b = 0

for prompt in test_prompts:
    print("="*30)
    print(f"USER PROMPT: {prompt}")

    # step 1: get responses fro both llms
    answer_a = get_llm_response(llm_a, prompt)
    answer_b = get_llm_response(llm_b, prompt)

    # step 2: judge the responses
    result = judge_responses(
        judge_llm=judge_llm,
        user_prompt=prompt,
        answer_a=answer_a,
        answer_b=answer_b
    )

    # step 3: diplay individual results
    print(f"Winner: {result["winner"]}")
    print(f"Reason: {result["reason"]}")

    if result["winner"] == "A":
        score_a += 1
    elif result["winner"] == "B":
        score_b += 1

# final results
print("="*30)
print("FINAL SCORES")
print(f"LLM A Score: {score_a}")
print(f"LLM B Score: {score_b}")

if score_a > score_b:
    print("Overall winner: LLM A")
elif score_b > score_a:
    print("Overall winner: LLM B")
else:
    print("Overall result: Tie")


USER PROMPT: Explain AWS IAM in simple terms
Winner: B
Reason: Both are broadly correct and beginner-friendly, but B is clearer and more accurate about how authorization works (IAM evaluates requests against policies and allows/denies). A has a misleading workflow implying you must add a user to a group to inherit a policy after attaching one, and its step-by-step process is less representative of real IAM usage. A also omits a key best practice (avoid using long-term user credentials; prefer roles), while B’s example correctly highlights roles for applications.
USER PROMPT: Difference between Security Group and NACL
Winner: B
Reason: Answer A contains multiple correctness errors: Security Group rules are not evaluated in order (there is no rule-order/first-match behavior), and NACLs do not have a default-allow policy by default (the default NACL allows all, but NACLs in general can be configured otherwise). Answer B is largely correct on stateful vs stateless and instance- vs subnet-s